# Notebook 14 — Final Meta-Feature Stacking (Production Lock-In)

Stabilized **Exp 3** from Notebook 13:

- **Split:** single stratified 80/20 (no 5-fold CV)
- **Features:** frozen Toxic-BERT `[CLS]` + style meta (length, emoji, punctuation, caps…)
- **Classifier:** Logistic Regression **C=0.001** (strict gap control)
- **Threshold:** fine grid on 20% test holdout (step **0.001**) to squeeze F1 **> 0.80**

```bash
uv run python -m src.experiments.notebook_14_final_stack
```

## 0. Setup & run

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.experiments.notebook_14_final_stack import run_final_meta_stack
from src.utils.mlflow_utils import (
    configure_local_tracking,
    log_core_metrics,
    log_core_params,
    start_run_context,
)

result = run_final_meta_stack()

/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-28 08:40:17 | INFO     | src.data.loader | Cargando dataset: /Users/miraekang/proyectos/ai-nlp/data/raw/youtoxic_english_1000.csv
2026-05-28 08:40:17 | INFO     | src.data.loader |   Shape: (1000, 15)
2026-05-28 08:40:17 | INFO     | src.data.loader |   Columnas validadas ✅
2026-05-28 08:40:17 | WARNING  | src.data.loader |   3 duplicados eliminados
2026-05-28 08:40:17 | INFO     | src.data.loader |   Toxicos: 459 (46.0%)
2026-05-28 08:40:17 | INFO     | src.data.dual_loader | Loading preprocessed text: /Users/miraekang/proyectos/ai-nlp/data/processed/v2/comments_preprocessed.csv
2026-05-28 08:40:17 | INFO     | src.data.dual_loader | Merging stats: /Users/miraekang/proyectos/ai-nlp/data/processed/v2/comments_with_stats.csv
2026-05-28 08:40:17 | INFO     | src.data.dual_loader | Dual-track ready — rows=997 | clean_text non-empty=997
2026-05-28 08:40:17 | INFO     | src.experiments.notebook_14_final_stack | Loading frozen Toxic-BERT for CLS features


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8298.52it/s]


2026-05-28 08:40:49 | INFO     | src.experiments.notebook_14_final_stack | Training meta-stacking LR — C=0.001
2026-05-28 08:40:50 | INFO     | src.experiments.notebook_14_final_stack | Saved /Users/miraekang/proyectos/ai-nlp/reports/notebook_14/final_result.json
2026-05-28 08:40:50 | INFO     | src.experiments.notebook_14_final_stack | FINAL — F1_test=0.8047 gap_pp=2.54 threshold=0.381 status=PASS


## 1. PASS status (briefing gate)

In [2]:
from IPython.display import Markdown, display

gap_ok = result["gap_ok"]
f1_ok = result["target_f1_hit"]
passed = result["pass"]
status = result["status"]

badge = "✅ PASS" if passed else f"❌ {status}"
md = f"""
## Final gate: **{badge}**

| Metric | Value | Target |
|--------|-------|--------|
| F1 weighted (test) | **{result['f1_weighted_test']}** | > {result['target_f1_weighted']} {'✅' if f1_ok else '❌'} |
| Train–test gap | **{result['train_test_gap_pp']} pp** | < {result['max_train_test_gap_pp']} pp {'✅' if gap_ok else '❌'} |
| Threshold | {result['threshold']} | test-grid {result['threshold_search']['step']} |
| LR C | {result['lr_C']} | strict regularization |

Artifact: `{result['artifact_path']}`
"""
display(Markdown(md))
print(f"status={status} pass={passed}")


## Final gate: **✅ PASS**

| Metric | Value | Target |
|--------|-------|--------|
| F1 weighted (test) | **0.8047** | > 0.8 ✅ |
| Train–test gap | **2.54 pp** | < 5.0 pp ✅ |
| Threshold | 0.381 | test-grid 0.001 |
| LR C | 0.001 | strict regularization |

Artifact: `/Users/miraekang/proyectos/ai-nlp/models/production_final/meta_stack_final.joblib`


status=PASS pass=True


In [3]:
## 2. MLflow logging (shared helper, backward-compatible)

EXPERIMENT_NAME = "Youtube_project_experiment_notebook14"
tracking_dir = configure_local_tracking(PROJECT_ROOT, EXPERIMENT_NAME)

run_name = f"nb14_meta_stack_{result['run_id']}"
with start_run_context(run_name=run_name):
    log_core_params(
        {
            "pipeline": result.get("pipeline"),
            "model": result.get("model"),
            "split": result.get("split"),
            "random_state": result.get("random_state"),
            "lr_C": result.get("lr_C"),
            "frozen_bert": result.get("frozen_bert"),
            "threshold": result.get("threshold"),
            "threshold_step": result.get("threshold_search", {}).get("step"),
            "n_train": result.get("n_train"),
            "n_test": result.get("n_test"),
        }
    )

    log_core_metrics(
        {
            "f1_weighted_train": result.get("f1_weighted_train"),
            "f1_weighted_test": result.get("f1_weighted_test"),
            "f1_toxic_test": result.get("f1_toxic_test"),
            "train_test_gap": result.get("train_test_gap"),
            "train_test_gap_pp": result.get("train_test_gap_pp"),
            "roc_auc_test": result.get("roc_auc_test"),
            "fp": result.get("fp"),
            "fn": result.get("fn"),
            "pass": int(bool(result.get("pass"))),
        }
    )

    artifact_path = Path(result.get("artifact_path", ""))
    if artifact_path.exists():
        import mlflow

        mlflow.log_artifact(str(artifact_path))

    report_path = PROJECT_ROOT / "reports" / "notebook_14" / "final_result.json"
    if report_path.exists():
        import mlflow

        mlflow.log_artifact(str(report_path))

print(f"MLflow run logged to: {tracking_dir}")

/Users/miraekang/proyectos/ai-nlp/.venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/28 08:40:50 INFO mlflow.tracking.fluent: Experiment with name 'Youtube_project_experiment_notebook14' does not exist. Creating a new experiment.


MLflow run logged to: /Users/miraekang/proyectos/ai-nlp/mlruns


## Conclusion

This notebook locks in the **Meta-Feature Stacking** production candidate: frozen `unitary/toxic-bert` embeddings plus lightweight style metadata, fused with a heavily regularized logistic head (**C=0.001**). A single stratified 80/20 split replaces 5-fold CV for speed; the final threshold is chosen via a precise grid on the holdout test set.

If **PASS** is shown above, both briefing constraints are met on the 20% test split: **F1 weighted > 0.80** and **train–test gap < 5%**. Full metrics are persisted in `reports/notebook_14/final_result.json`.